## **0. Tải bộ dữ liệu**
**Lưu ý:** Nếu bạn không thể sử dụng lệnh gdown để tải bộ dữ liệu vì bị giới hạn số lượt tải, hãy tải bộ dữ liệu thử công và upload lên google drive của mình. Sau đó, sử dụng lệnh dưới đây để copy file dữ liệu vào colab:
```python
from google.colab import drive

drive.mount('/content/drive')
!cp /path/to/dataset/on/your/drive .
```

In [ ]:
# https://drive.google.com/file/d/1yuAdAo3Gex-HsE1hB3Fa1hL5ucew4Jtb/view?usp=sharing
!gdown 1yuAdAo3Gex-HsE1hB3Fa1hL5ucew4Jtb

## **1. Import các thư viện cần thiết**

In [1]:
import string
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


## **2. Đọc bộ dữ liệu**

In [2]:
DATASET_PATH = '/content/cls_spam_text.csv'
df = pd.read_csv(DATASET_PATH)
df

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [3]:
messages = df['Message'].values.tolist()
labels = df['Category'].values.tolist()

## **3. Chuẩn bị bộ dữ liệu**

### **3.1. Xử lý dữ liệu nhãn**

In [6]:
# Xử lý dữ liệu LabelEncoder
le = LabelEncoder()
y = le.fit_transform(labels)
print(f'Classes: {le.classes_}')
print(f'Encoded labels: {y}')

Classes: ['ham' 'spam']
Encoded labels: [0 0 1 ... 0 0 0]


### **3.2. Xử lý dữ liệu đặc trưng**

In [8]:
def lowercase(text):
    # Code ở đây
    return text.lower()

def punctuation_removal(text):
    # Code ở đây
    translator = str.maketrans('', '', string.punctuation)

    return text.translate(translator)
def tokenize(text):
    # Code ở đây
    return nltk.word_tokenize(text)

def remove_stopwords(tokens):
    # Code ở đây
    stop_words = nltk.corpus.stopwords.words('english')

    return [token for token in tokens if token not in stop_words]


def stemming(tokens):
    # Code ở đây
    stemmer = nltk.stem.PorterStemmer()

    return [stemmer.stem(token) for token in tokens]


def preprocess_text(text):
    # Code ở đây
    text = lowercase(text)
    text = punctuation_removal(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = stemming(tokens)

    return tokens

In [9]:
messages = [
    preprocess_text(message) for message in messages
]

In [10]:
def create_dictionary(messages):
    # Code ở đây
    dictionary = set()
    for tokens in messages:
        for token in tokens:
            dictionary.add(token)

    return dictionary


def create_features(tokens, dictionary):
    # Code ở đây
    features = np.zeros(len(dictionary))

    for token in tokens:
        if token in dictionary:
            index = list(dictionary).index(token)
            features[index] += 1

    return features

In [11]:
dictionary = create_dictionary(messages)
X = np.array(
    [create_features(tokens, dictionary) for tokens in messages]
)

In [12]:
X[0]

array([0., 0., 0., ..., 0., 0., 0.])

### **3.3. Chia dữ liệu train/val/test**

In [13]:
VAL_SIZE = 0.2
TEST_SIZE = 0.125
SEED = 0

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=VAL_SIZE,
    shuffle=True,
    random_state=SEED
)
X_train, X_test, y_train, y_test = train_test_split(
    X_train, y_train,
    test_size=TEST_SIZE,
    shuffle=True,
    random_state=SEED
)

## **4. Huấn luyện mô hình**

In [14]:
%%time
model = MultinomialNB()
print('Start training...')
model = model.fit(X_train, y_train)
print('Training completed!')

Start training...
Training completed!
CPU times: user 198 ms, sys: 885 µs, total: 199 ms
Wall time: 144 ms


## **5. Đánh giá mô hình**

In [15]:
y_val_pred = model.predict(X_val)
y_test_pred = model.predict(X_test)
val_accuracy = accuracy_score(y_val, y_val_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f'Val accuracy: {val_accuracy}')
print(f'Test accuracy: {test_accuracy}')

Val accuracy: 0.9820627802690582
Test accuracy: 0.9802867383512545


## **6. Thực hiện dự đoán**

In [16]:
def predict(text, model, dictionary):
    processed_text = preprocess_text(text)
    features = create_features(text, dictionary)
    features = np.array(features).reshape(1, -1)
    prediction = model.predict(features)
    prediction_cls = le.inverse_transform(prediction)[0]

    return prediction_cls

In [17]:
test_input = 'I am actually thinking a way of doing something useful'
prediction_cls = predict(test_input, model, dictionary)
print(f'Prediction: {prediction_cls}')

Prediction: ham
